In [ ]:
import time
import joblib
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

BASE = Path("/Users/limhyunjin/Desktop/mission18/backend")

MODEL_PATH = BASE / "data" / "sentiment_model.joblib"
df = pd.read_csv(BASE / "data" / "nsmc_sample.csv").dropna()

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
print(f"학습 {len(train):,}건 / 평가 {len(test):,}건")
print(train[["document", "label"]].head(3).to_string(index=False))

학습 29,999건 / 평가 5,000건
                         document  label
              아 더빙.. 진짜 짜증나네요 목소리      0
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
                너무재밓었다그래서보는것을추천한다      0


In [2]:
fit_train, valid = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=42,
)

print(
    f"학습 {len(fit_train):,}건 | "
    f"검증 {len(valid):,}건 | "
    f"최종 평가 {len(test):,}건"
)

학습 23,999건 | 검증 6,000건 | 최종 평가 5,000건


In [4]:
model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4),
                    min_df=3, max_features=50000, sublinear_tf=True),
    LogisticRegression(C=4, max_iter=1000),
)

t0 = time.time()
model.fit(train["document"], train["label"])
fit_sec = time.time() - t0

acc = model.score(test["document"], test["label"])
joblib.dump(model, MODEL_PATH, compress=3)
size_mb = os.path.getsize(MODEL_PATH) / 1024**2

print(f"학습 {fit_sec:.1f}초 | 정확도 {acc:.4f} | 모델 파일 {size_mb:.2f} MB")


학습 1.6초 | 정확도 0.8480 | 모델 파일 0.82 MB


In [5]:
print(model.classes_)

[0 1]


In [6]:
for text in ["연출도 좋고 배우 연기가 정말 훌륭했어요", "시간이 아깝다. 최악",
             "그냥 그랬음", "기대 이상! 강추합니다", "스토리가 너무 지루하고 뻔했다"]:
    p = model.predict_proba([text])[0][1]
    print(f"{text:26} {'긍정' if p >= 0.5 else '부정'}  {p:.3f}")


연출도 좋고 배우 연기가 정말 훌륭했어요     긍정  0.989
시간이 아깝다. 최악                부정  0.000
그냥 그랬음                     부정  0.094
기대 이상! 강추합니다               긍정  0.727
스토리가 너무 지루하고 뻔했다           부정  0.006


In [7]:
# 피쳐 수별 검증 정확도, 학습 시간, 모델 파일 크기 변환
def compare_model(max_features: int) -> dict:
    candidate = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4),
                    min_df=3, max_features=max_features, sublinear_tf=True),
    LogisticRegression(C=4, max_iter=1000),
    )   
    t0 = time.perf_counter()
    candidate.fit(fit_train["document"], fit_train["label"])
    fit_sec = time.perf_counter() - t0

    valid_acc = candidate.score(valid["document"], valid["label"])
    comparsion_path = BASE / "data" / f"compare_{max_features}.joblib"
    joblib.dump(candidate, comparsion_path, compress=3)
    size_mb = os.path.getsize(comparsion_path) / 1024**2

    return {
        "max_features": max_features,
        "valid_accuracy": valid_acc,
        "fit_sec": fit_sec,
        "size_mb": size_mb,
    }

In [8]:
results = [
    compare_model(50000),
    compare_model(20000),
]

pd.DataFrame(results)

,max_features,valid_accuracy,fit_sec,size_mb
0,50000,0.847333,1.368123,0.816109
1,20000,0.839167,1.172320,0.331360


In [9]:
pd.DataFrame(results).to_csv(
    BASE / "data" / "model_comparison.csv",
    index=False,
)